# LWSCE pilot on Colab

CE vs LWSCE on CIFAR100: each loss picks its learning rate on the validation set, then test accuracy is compared.

1. **Runtime → Change runtime type → GPU** (T4 is enough).
2. **Runtime → Run all.** Google Drive will ask for permission once.
3. Results and models are saved to Google Drive (`MyDrive/label_mask_loss_runs`). If the session disconnects, just **Run all** again: finished loss × lr combinations are skipped (a combination that was interrupted midway restarts from its first run).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
print("torch", torch.__version__)

## 1. Experiment settings

In [ ]:
# where the code comes from: "github" clones BRANCH, "zip" asks you to upload a zip of the repo
CODE_SOURCE = "github"
REPO_URL = "https://github.com/Frank-Jin54/label_mask_loss.git"
BRANCH = "revert"

DATASET = "CIFAR100"
OPT_ALG = "ADAM"
LOSSES = ["CROSSENTROPY", "LWSCE"]
LRS = ["1e-3", "3e-4", "1e-4"]
RUNS = 3
EPOCH = 31          # training.py trains EPOCH - 1 epochs
ALPHA = 0.2

DRIVE_DIR = "/content/drive/MyDrive/label_mask_loss_runs"
CODE_DIR = "/content/label_mask_loss"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(os.path.join(DRIVE_DIR, "result"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, "model"), exist_ok=True)

## 3. Get the code

In [ ]:
import os, shutil, subprocess, zipfile

if CODE_SOURCE == "github":
    if os.path.exists(CODE_DIR):
        subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
                        REPO_URL, CODE_DIR], check=True)
elif CODE_SOURCE == "zip":
    from google.colab import files
    uploaded = files.upload()  # a zip of the repo containing training.py, pilot.py, loss/, model_define/, cv/
    zip_name = next(iter(uploaded))
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(CODE_DIR)
    # accept a zip whose content sits inside one top-level folder
    entries = os.listdir(CODE_DIR)
    if "training.py" not in entries and len(entries) == 1:
        inner = os.path.join(CODE_DIR, entries[0])
        for name in os.listdir(inner):
            shutil.move(os.path.join(inner, name), CODE_DIR)
        os.rmdir(inner)
else:
    raise ValueError(CODE_SOURCE)

for f in ["training.py", "pilot.py", "loss/partial_loss.py"]:
    assert os.path.exists(os.path.join(CODE_DIR, f)), "missing " + f + ", is the branch / zip up to date?"

# keep results and models on Drive so they survive a disconnect
for name in ["result", "model"]:
    path = os.path.join(CODE_DIR, name)
    if os.path.islink(path):
        continue
    shutil.rmtree(path, ignore_errors=True)
    os.symlink(os.path.join(DRIVE_DIR, name), path)

%cd {CODE_DIR}
!git log --oneline -1 2>/dev/null || true

## 4. Run the pilot

About 6 × 3 × 30 epochs; on a T4 expect a few hours. Output streams below.

In [ ]:
losses = " ".join(LOSSES)
lrs = " ".join(LRS)
!PYTHONUNBUFFERED=1 python pilot.py --dataset {DATASET} --opt_alg {OPT_ALG} --losses {losses} --lrs {lrs} --runs {RUNS} --epoch {EPOCH} --alpha {ALPHA}

## 5. Summary

The lr marked `*` is chosen by validation accuracy; compare `test_acc` across losses.

In [ ]:
losses = " ".join(LOSSES)
lrs = " ".join(LRS)
!python pilot.py --summarize_only --dataset {DATASET} --opt_alg {OPT_ALG} --losses {losses} --lrs {lrs} --runs {RUNS} --alpha {ALPHA}

## 6. Download results (optional)

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/pilot_results", "zip", os.path.join(DRIVE_DIR, "result"))
files.download(archive)